In [ ]:
# 1. Setup và Load dữ liệu
import pandas as pd
import sys
import os
sys.path.append(os.path.abspath("../src"))
from apriori_library import DataCleaner, BasketPreparer, AssociationRulesMiner

# Load và clean data
cleaner = DataCleaner("../data/raw/online_retail.csv")
df = cleaner.load_data()
df_uk = cleaner.clean_data()

In [ ]:
# 2. TÍNH TRỌNG SỐ (Bước quan trọng nhất của Subject 3)
# Tính tổng tiền cho mỗi hóa đơn (InvoiceNo)
# df_uk cần có cột TotalPrice (Quantity * UnitPrice)
if "TotalPrice" not in df_uk.columns:
    df_uk["TotalPrice"] = df_uk["Quantity"] * df_uk["UnitPrice"]

# Tạo Series trọng số: Index là InvoiceNo, Value là tổng tiền
invoice_weights = df_uk.groupby("InvoiceNo")["TotalPrice"].sum()
print(f"Tổng doanh thu toàn bộ: {invoice_weights.sum():,.2f}")

In [ ]:
# 3. Chuẩn bị Basket và chạy Apriori như thường
preparer = BasketPreparer(df_uk)
basket = preparer.create_basket()
basket_bool = preparer.encode_basket()

miner = AssociationRulesMiner(basket_bool)
miner.mine_frequent_itemsets(min_support=0.01) # Ngưỡng support thường
miner.generate_rules(metric="lift", min_threshold=1.0)
miner.add_readable_rule_str()


In [ ]:
# 4. TÍNH WEIGHTED METRICS (Gọi hàm vừa thêm)
# Truyền invoice_weights vào hàm
miner.calculate_weighted_metrics(weights_series=invoice_weights)


In [ ]:
# 5. So sánh và Phân loại
rules = miner.rules
print(rules[['rule_str', 'support', 'weighted_support', 'lift']].head())
# Lọc ra các nhóm luật
# Nhóm "Ngôi sao": Support cao (>0.02) và Weighted Support cao (>0.02)
stars = rules[(rules['support'] > 0.02) & (rules['weighted_support'] > 0.02)]

# Nhóm "Giá trị cao" (Niche): Support thấp (<0.02) nhưng Weighted Support cao (>0.01)
niche_value = rules[(rules['support'] < 0.02) & (rules['weighted_support'] > 0.01)]

print(f"Số lượng luật Ngôi sao: {len(stars)}")
print(f"Số lượng luật Giá trị cao (Niche): {len(niche_value)}")